# 🚀 Phase 2A: Track A Few-Shot & Zero-Day Generalization Benchmark ($N \le 10\text{k}$)
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection: An Axiomatic-Empirical Fuzzy DEMATEL Simulation Framework*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

---

### 📌 Track A Scientific Benchmark Objectives:
1. **Multi-Paradigm Comparative Evaluation**: Benchmark 8 modern AI architectures across 5-fold cross-validation:
   - **Foundation Models**: `TabPFN v3`, `TabICL v2` (Bayesian In-Context Zero-Shot Learning without backprop)
   - **Modern Deep Learning**: `Mambular SSM` ($O(L)$ linear state-space), `FT-Transformer` ($O(L^2)$ attention), `SAINT` (dual self/row attention), `GraphIDS` (Inductive GNN)
   - **Tuned Baselines**: `XGBoost`, `LightGBM` (Optuna 50-trial bayesian optimization)
2. **Authentic Dataset Binding**: Ingest decontaminated data directly from `data/processed/CICIDS2017_cleaned.parquet` (or authentic raw flows in `data/raw/MachineLearningCVE/`).
3. **Zero-Day Holdout Generalization Protocol**: Evaluate unseen attack detection capability by holding out rare attack families during training to measure $F_{1\text{-unseen}}$.
4. **Anti-Leakage Protection**: Strict `GroupKFold` partitioning by `/24` subnet masks with fold-isolated preprocessing.
5. **Autorecovery Checkpointing**: Powered by `CheckpointManager` on Google Drive—survives sudden Colab disconnects and preemptions without restarting from fold 1.
6. **Task-Technology Fit (TTF) Multi-Task Utility Analysis**: Operationalize deployment fitness for:
   - **$T_1$ (High-Speed Edge / Core Transit)**: Prioritizing throughput $\ge 100\text{k flows/s}$ and sub-millisecond latency.
   - **$T_2$ (Enterprise Threat Intelligence)**: Prioritizing Macro $F_1 \ge 0.95$ and Zero-Day discovery.
   - **$T_3$ (Resource-Constrained Gateway / IoT Hub)**: Prioritizing peak VRAM $\le 500\text{MB}$ and low CPU footprint.


### 1. ☁️ Google Drive Mount & Project Root Auto-Resolution


In [ ]:
import os, sys
from pathlib import Path

# 0. Enable automatic reloading of modified modules in Colab / Jupyter
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# 1. Mount Google Drive if running inside Colab
try:
    from google.colab import drive
    if not Path('/content/drive').exists() and not Path('/content/My Drive').exists():
        drive.mount('/content/drive')
except ImportError:
    print("ℹ️ Running in local/workstation environment.")

# 2. Candidate root paths (supporting both 'Colab Notebook' and 'Colab Notebooks')
CANDIDATE_ROOTS = [
    Path('/content/drive/MyDrive/Colab Notebooks'),
    Path('/content/drive/My Drive/Colab Notebooks'),
    Path('/content/drive/MyDrive/Colab Notebook'),
    Path('/content/drive/My Drive/Colab Notebook'),
    Path('/Colab Notebooks'),
    Path('/Colab Notebook'),
    Path('/content/My Drive/Colab Notebooks'),
    Path('/content/My Drive/Colab Notebook'),
    Path('/content/Colab Notebooks'),
    Path('/content/Colab Notebook'),
    Path('/content/drive/MyDrive/is_ai-vuln'),
    Path('/content/drive/My Drive/is_ai-vuln'),
    Path('/content/is_ai-vuln'),
    Path('.').resolve()
]

PROJECT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists() and ((cand / 'src').exists() or (cand / 'data' / 'raw').exists()):
        PROJECT_ROOT = cand.resolve()
        break

# Dynamic discovery inside /content/drive if not yet matched
if PROJECT_ROOT is None and Path('/content/drive').exists():
    for drive_parent in [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive'), Path('/content/drive'), Path('/content/My Drive')]:
        if drive_parent.exists():
            try:
                for sub in drive_parent.iterdir():
                    if sub.is_dir() and ('colab notebook' in sub.name.lower() or 'is_ai-vuln' in sub.name.lower()):
                        if (sub / 'src').exists() or (sub / 'data' / 'raw').exists():
                            PROJECT_ROOT = sub.resolve()
                            break
            except Exception:
                pass
            if PROJECT_ROOT:
                break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.').resolve()

os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. Locate authentic dataset raw storage directory across candidate paths
DATA_RAW_DIR = None
KNOWN_SUBFOLDERS = ['cic-ddos2019', 'machinelearningcve', 'nsl-kdd', 'ton-iot', 'trafficlabelling', 'unsw-data-full']

for cand_raw in [
    Path('/content/drive/MyDrive/Colab Notebooks/data/raw'),
    Path('/content/drive/My Drive/Colab Notebooks/data/raw'),
    Path('/content/drive/MyDrive/Colab Notebook/data/raw'),
    Path('/content/drive/My Drive/Colab Notebook/data/raw'),
    Path('/Colab Notebooks/data/raw'),
    Path('/Colab Notebook/data/raw'),
    PROJECT_ROOT / 'src' / 'data' / 'actual-data',
    PROJECT_ROOT / 'actual-data',
    PROJECT_ROOT / 'data' / 'raw',
]:
    if cand_raw.exists():
        try:
            sub_names = [c.name.lower() for c in cand_raw.iterdir() if c.is_dir()]
            if any(k in sub_names for k in KNOWN_SUBFOLDERS):
                DATA_RAW_DIR = cand_raw.resolve()
                break
        except Exception:
            pass

# Dynamic search inside /content/drive if not yet matched
if DATA_RAW_DIR is None and Path('/content/drive').exists():
    for drive_parent in [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive'), Path('/content/drive'), Path('/content/My Drive')]:
        if drive_parent.exists():
            try:
                for sub in drive_parent.iterdir():
                    if sub.is_dir() and ('colab notebook' in sub.name.lower() or 'is_ai-vuln' in sub.name.lower()):
                        cand = sub / 'data' / 'raw'
                        if cand.exists():
                            sub_names = [c.name.lower() for c in cand.iterdir() if c.is_dir()]
                            if any(k in sub_names for k in KNOWN_SUBFOLDERS):
                                DATA_RAW_DIR = cand.resolve()
                                break
            except Exception:
                pass
            if DATA_RAW_DIR:
                break

if DATA_RAW_DIR is None:
    DATA_RAW_DIR = (PROJECT_ROOT / 'data' / 'raw').resolve()

# Link local data/raw to Drive data/raw if different (Colab Linux filesystem)
local_raw = PROJECT_ROOT / 'data' / 'raw'
if DATA_RAW_DIR.exists() and local_raw.resolve() != DATA_RAW_DIR.resolve():
    if not local_raw.exists():
        try:
            local_raw.parent.mkdir(parents=True, exist_ok=True)
            local_raw.symlink_to(DATA_RAW_DIR, target_is_directory=True)
            print(f"🔗 Linked: {local_raw} -> {DATA_RAW_DIR}")
        except Exception:
            pass

detected_folders = [f.name for f in DATA_RAW_DIR.iterdir() if f.is_dir()] if DATA_RAW_DIR.exists() else []

print("=" * 80)
print(f"✅ Active Project Root : {PROJECT_ROOT}")
print(f"✅ src/ directory found: {(PROJECT_ROOT / 'src').exists()}")
print(f"📁 Active Raw Data Path: {DATA_RAW_DIR}")
print(f"🔍 Detected Raw Folders: {detected_folders}")
known_real = ['CIC-DDoS2019', 'MachineLearningCVE', 'NSL-KDD', 'TON-IoT', 'ToN-IOT', 'TrafficLabelling', 'unsw-data-full']
found_known = [f for f in detected_folders if f in known_real]
if found_known:
    print(f"🛡️ [DATA STATUS: REAL BENCHMARK DATASETS DETECTED] Found: {found_known}")
else:
    print("ℹ️ [DATA STATUS] Datasets will be dynamically located across Drive paths.")
print("=" * 80)


### 2. 📦 Core & Model Dependencies Installation


In [ ]:
# Install core dependencies for Track A benchmark
!pip install -q xgboost lightgbm optuna scikit-learn imbalanced-learn pandas numpy matplotlib seaborn networkx requests tqdm

print("✅ Benchmark dependencies ready.")


### 3. 🛡️ Data Ingestion & Authentic Benchmark Data Loading

Loads the decontaminated benchmark dataset from Phase 1 (`data/processed/CICIDS2017_cleaned.parquet`), or automatically runs ingestion on the authentic raw files from Google Drive (`data/raw/MachineLearningCVE/`).


In [ ]:
import os
from pathlib import Path
import pandas as pd
from src.data.drive_downloader import initialize_dataset_directories

dirs = initialize_dataset_directories(PROJECT_ROOT)
clean_file = dirs["processed"] / "CICIDS2017_cleaned.parquet"
if not clean_file.exists():
    clean_file = dirs["processed"] / "CICIDS2017_cleaned.csv"

if clean_file.exists():
    print(f"🛡️ [DATA STATUS: LOADING DECONTAMINATED DATASET]")
    print(f"📁 Source: {clean_file.resolve()}")
    df_benchmark = pd.read_parquet(clean_file) if str(clean_file).endswith(".parquet") else pd.read_csv(clean_file)
    is_synthetic_data = False
else:
    print("🔄 Decontaminated file not found in processed/. Ingesting directly from authentic raw storage...")
    # Search authentic folders directly
    raw_search_dirs = [
        DATA_RAW_DIR / "MachineLearningCVE" if DATA_RAW_DIR else None,
        PROJECT_ROOT / "data" / "raw" / "MachineLearningCVE",
        PROJECT_ROOT / "src" / "data" / "actual-data" / "MachineLearningCVE",
        PROJECT_ROOT / "actual-data" / "MachineLearningCVE",
        Path("/content/drive/MyDrive/Colab Notebooks/data/raw/MachineLearningCVE"),
        Path("/content/drive/My Drive/Colab Notebooks/data/raw/MachineLearningCVE")
    ]
    raw_file = None
    for d in raw_search_dirs:
        if d and Path(d).exists():
            cand = Path(d) / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
            if cand.exists():
                raw_file = cand
                break
            files = [f for f in Path(d).iterdir() if f.is_file() and f.suffix.lower() == ".csv" and not f.name.startswith(".")]
            if files:
                raw_file = files[0]
                break
    if raw_file:
        print(f"📄 Found authentic raw file: {raw_file}")
        df_raw = pd.read_csv(raw_file)
        try:
            from src.data.cleaner import clean_dataset
            df_benchmark, _ = clean_dataset(df_raw, "CICIDS2017")
        except Exception:
            df_benchmark = df_raw.dropna().drop_duplicates()
        dirs["processed"].mkdir(parents=True, exist_ok=True)
        try:
            df_benchmark.to_parquet(clean_file, index=False)
        except Exception:
            df_benchmark.to_csv(dirs["processed"] / "CICIDS2017_cleaned.csv", index=False)
        is_synthetic_data = False
    else:
        print("⚠️ Authentic raw file not found. Generating minimal temporary sample for validation...")
        from src.data.drive_downloader import generate_synthetic_benchmark_sample
        sample_path = dirs["raw"] / "CICIDS2017_sample.csv"
        generate_synthetic_benchmark_sample("CICIDS2017", sample_path, n_samples=5000)
        df_benchmark = pd.read_csv(sample_path)
        is_synthetic_data = True

print(f"📊 Active Benchmark NetFlows: {len(df_benchmark):,} records across {len(df_benchmark.columns)} features.")
print(f"🛡️ Provenance: {'SYNTHETIC FALLBACK' if is_synthetic_data else 'REAL AUTHENTIC NETFLOW BENCHMARK'}")


### 4. 🔄 Fault-Tolerant Checkpoint Bootstrap (`CheckpointManager`)


In [ ]:
import json
from src.utils.checkpoint_manager import CheckpointManager

# Establish checkpoint directory on Google Drive
chk_dir = PROJECT_ROOT / "checkpoints"
chk_dir.mkdir(parents=True, exist_ok=True)

manager = CheckpointManager(
    drive_checkpoint_dir=chk_dir,
    dataset_name="CICIDS2017",
    track_name="Track_A",
    total_folds=5
)

print(f"🔄 Checkpoint Status   : {manager.state['status']}")
print(f"✅ Completed Models    : {manager.state['completed_models']}")
print(f"▶️ Active Resume Target: Model '{manager.state['current_model']}' (Fold {manager.state['current_fold']})")


### 5. 🔬 Track A 5-Fold Cross-Validation Execution (8 Architectures)

Executes the complete cross-validation loop across:
1. `TabPFN_v3` (Bayesian In-Context Prior Network)
2. `TabICL_v2` (Tabular In-Context Attention)
3. `Mambular_SSM` (Linear State-Space Model $O(L)$)
4. `FT_Transformer` (Feature Tokenizer Transformer $O(L^2)$)
5. `SAINT` (Dual Self & Intersample Attention)
6. `GraphIDS` (Inductive GNN Message Passing)
7. `XGBoost` (Hist-Gradient Boosting)
8. `LightGBM` (Exclusive Feature Bundling GBDT)


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from src.models import get_model
from src.evaluation import evaluate_fold_run, calculate_ttf_utility
from src.data.splitters import AntiLeakageGroupKFold, extract_subnet_mask, safe_slice
from src.utils.environment import flush_memory

# Sample N=10,000 records for Track A Few-Shot benchmark
df_track_a = df_benchmark.head(10000).copy()
target_col = "is_attack" if "is_attack" in df_track_a.columns else df_track_a.columns[-1]
feature_cols = [c for c in df_track_a.select_dtypes(include=[np.number]).columns if c != target_col]

X = df_track_a[feature_cols].values
y = df_track_a[target_col].values

# Extract subnet blocks for host session leakage protection
src_ip_col = "source_ip" if "source_ip" in df_track_a.columns else None
subnets = extract_subnet_mask(df_track_a[src_ip_col]) if src_ip_col else pd.Series(np.arange(len(df_track_a)) // 2000)

models_to_run = [
    "XGBoost",
    "LightGBM",
    "TabPFN_v3",
    "TabICL_v2",
    "Mambular_SSM",
    "FT_Transformer",
    "SAINT",
    "GraphIDS"
]

gkf = AntiLeakageGroupKFold(n_splits=5)
fold_indices = list(gkf.split(X, y, groups=subnets))

all_metrics = []

for model_name in models_to_run:
    if manager.should_skip_model(model_name):
        print(f"⏩ Model '{model_name}' already fully completed in checkpoint. Skipping...")
        continue
        
    print(f"\n{'='*60}\n🚀 Running Architecture: {model_name}\n{'='*60}")
    
    for fold_idx, (train_idx, val_idx) in enumerate(fold_indices, start=1):
        if manager.should_skip_fold(model_name, fold_idx):
            print(f"  ⏩ Fold {fold_idx} already cached. Skipping...")
            continue
            
        print(f"  ▶️ Training {model_name} | Fold {fold_idx}/5 (Train: {len(train_idx):,}, Val: {len(val_idx):,})...")
        
        # 1. Fold-isolated scaling
        X_tr, y_tr = safe_slice(X, train_idx), safe_slice(y, train_idx)
        X_va, y_va = safe_slice(X, val_idx), safe_slice(y, val_idx)
        
        scaler = StandardScaler().fit(X_tr)
        X_tr_sc = scaler.transform(X_tr)
        X_va_sc = scaler.transform(X_va)
        
        # 2. Fit Model
        model = get_model(model_name)
        t_fit_start = time.perf_counter()
        model.fit(X_tr_sc, y_tr)
        train_duration = time.perf_counter() - t_fit_start
        
        # 3. Predict & Profile Inference
        preds = model.predict(X_va_sc)
        probs = model.predict_proba(X_va_sc)
        profile = model.profile_inference(X_va_sc, warmup_runs=3, repeat_runs=10)
        
        # 4. Compute Metrics & Multi-Task TTF Utility
        fold_met = evaluate_fold_run(y_va, preds, probs, profile)
        fold_met["train_time_sec"] = round(train_duration, 3)
        fold_met["ttf_t1"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T1")
        fold_met["ttf_t2"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T2")
        fold_met["ttf_t3"] = calculate_ttf_utility(fold_met["f1_macro"], fold_met["latency_ms_per_flow"], 0.05, fold_met["vram_peak_mb"], 10.0, task="T3")
        
        manager.save_fold_progress(
            model_name=model_name,
            fold_index=fold_idx,
            metrics=fold_met,
            predictions=preds,
            probabilities=probs
        )
        all_metrics.append({"model": model_name, "fold": fold_idx, **fold_met})
        print(f"  ✅ Fold {fold_idx} Complete | Macro F1: {fold_met['f1_macro']:.4f} | Latency: {fold_met['latency_ms_per_flow']:.3f}ms | TTF(T1): {fold_met['ttf_t1']:.3f}")
        flush_memory()

# Save compiled benchmark results for downstream phases
output_dir = PROJECT_ROOT / "experiment_output" / "track_a"
output_dir.mkdir(parents=True, exist_ok=True)

df_results = pd.DataFrame(all_metrics)
if not df_results.empty:
    summary = df_results.groupby("model").agg({
        "f1_macro": ["mean", "std"],
        "f1_unseen": ["mean", "std"],
        "latency_ms_per_flow": ["mean"],
        "throughput_flows_sec": ["mean"],
        "vram_peak_mb": ["mean"],
        "ttf_t1": ["mean"],
        "ttf_t2": ["mean"],
        "ttf_t3": ["mean"]
    })
    summary.to_csv(output_dir / "master_summary.csv")
    with open(output_dir / "benchmark_results.json", "w", encoding="utf-8") as f:
        json.dump({
            "is_synthetic": is_synthetic_data,
            "dataset": "CICIDS2017",
            "models": models_to_run,
            "metrics": all_metrics
        }, f, indent=2)
    print(f"\n💾 Master benchmark summary saved to: {output_dir / 'master_summary.csv'}")
    print(f"💾 Telemetry JSON serialized to      : {output_dir / 'benchmark_results.json'}")
    display(summary)


### 6. 📊 Publication-Quality Pareto Frontiers & TTF Utility Visualizer

Renders comparative visualizations contrasting:
1. Macro F1 vs. Unseen Attack F1 (Zero-Day generalizability)
2. Macro F1 vs. Latency Pareto trade-off frontier


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(all_metrics) > 0:
    df_m = pd.DataFrame(all_metrics)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=140)
    
    # Plot 1: Macro F1 across architectures
    sns.barplot(data=df_m, x="model", y="f1_macro", palette="mako", ax=ax1, capsize=0.1)
    ax1.set_title(f"Track A: Macro F1 Score (Data: {'SYNTHETIC' if is_synthetic_data else 'REAL AUTHENTIC'})")
    ax1.set_ylabel("Macro F1 (5-Fold CV)")
    ax1.tick_params(axis='x', rotation=35)
    ax1.grid(True, linestyle="--", alpha=0.3)
    
    # Plot 2: Pareto Frontier: Accuracy vs Latency
    mean_perf = df_m.groupby("model").agg({"f1_macro": "mean", "latency_ms_per_flow": "mean"}).reset_index()
    sns.scatterplot(data=mean_perf, x="latency_ms_per_flow", y="f1_macro", hue="model", s=200, ax=ax2, palette="tab10")
    for _, row in mean_perf.iterrows():
        ax2.annotate(row["model"], (row["latency_ms_per_flow"] * 1.05, row["f1_macro"]), fontsize=9)
    ax2.set_title("Pareto Frontier: Accuracy vs. Per-Flow Latency")
    ax2.set_xlabel("Latency (ms / flow) [Log Scale, Lower is Better]")
    ax2.set_ylabel("Macro F1 Score [Higher is Better]")
    ax2.set_xscale("log")
    ax2.grid(True, linestyle="--", alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / "figure1_track_a_pareto.png")
    plt.show()
